# Détection intelligente des connexions anormales par IA
## 1. Introduction
SentinelAI est un prototype pédagogique local. Toutes les connexions sont fictives.
## 2. Objectifs
Construire un historique causal, apprendre une représentation du comportement ordinaire,
mesurer les erreurs et comparer Isolation Forest à des règles explicites.

## 3. Bibliothèques
Le notebook réutilise exactement les fonctions de l’application.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import plotly.express as px
from src.utils import DATA, ARTIFACTS
from src.generate_data import main as generate
from src.preprocessing import engineer_history, feature_matrix
from src.train_model import train
from src.predict import predict
from src.evaluate_model import evaluate, sensitivity

## 4. Chargement
Graine 42, 12 000 événements, 400 identifiants fictifs ; génération seulement si nécessaire.

In [ ]:
generate()
logs = pd.read_csv(DATA / 'login_logs.csv', parse_dates=['timestamp'])
logs.head()

## 5. Exploration
Les étiquettes décrivent les scénarios du simulateur, pas les décisions du modèle.

In [ ]:
display(logs.shape, logs.is_anomaly.value_counts(), logs.anomaly_type.value_counts())
px.histogram(logs, x='login_hour', color='is_anomaly').show()

## 6. Nettoyage
Les champs bruts obligatoires invalides sont rejetés ; les valeurs numériques manquantes des caractéristiques sont imputées par une médiane apprise sur le train uniquement.

In [ ]:
display(logs.isna().sum(), logs.event_id.duplicated().sum())
clean = engineer_history(logs)
assert len(clean) == len(logs)

## 7. Extraction des variables
Heure cyclique : sin(2πh/24), cos(2πh/24). Compteurs sur [t−10 min, t[, donc sans événement actuel. Nouveauté par utilisateur selon le passé disponible. Aucun identifiant, IP brute ou label n’est fourni au modèle.

In [ ]:
X = feature_matrix(clean)
display(X.head(), X.columns.tolist())
assert 'is_anomaly' not in X.columns

## 8. Découpage chronologique
75 % pour apprendre, 25 % pour évaluer. Le contexte antérieur du train est conservé pour les premières connexions du test : cette information est disponible dans un système réel.

In [ ]:
train_logs = pd.read_csv(DATA / 'train_logs.csv', parse_dates=['timestamp'])
test_logs = pd.read_csv(DATA / 'test_logs.csv', parse_dates=['timestamp'])
assert train_logs.timestamp.max() <= test_logs.timestamp.min()
display(len(train_logs), len(test_logs))

## 9. Apprentissage Isolation Forest
200 arbres, sous-échantillons de 256, contamination fixée à 8 %. L’entraînement utilise tout le train, majoritairement normal, sans sélectionner les lignes avec leurs labels. Les points isolés rapidement sont plus atypiques.

In [ ]:
bundle = train(train_logs, ARTIFACTS / 'notebook_model.joblib')

## 10. Prédiction
Scikit-learn renvoie −1 pour un outlier et +1 pour un inlier. Le dashboard les convertit en 1 et 0. Un score −decision_function positif indique un événement suspect ; ce score n’est pas une probabilité.

In [ ]:
scored = predict(test_logs, bundle)
scored[['event_id', 'prediction', 'anomaly_score', 'relative_percentile']].head()

## 11. Évaluation
Précision = VP/(VP+FP), rappel = VP/(VP+FN), F1 = moyenne harmonique. L’accuracy nécessite de tenir compte du déséquilibre. ROC-AUC et Average Precision utilisent le score continu.

In [ ]:
results = evaluate(test_logs, bundle)
display(pd.DataFrame(results).T.drop(columns='confusion_matrix'))

## 12. Graphiques
Matrice : lignes = réalité, colonnes = prédiction ; ordre normal puis suspect.

In [ ]:
px.imshow(results['Isolation Forest']['confusion_matrix'], text_auto=True, x=['Normal prédit','Suspect prédit'], y=['Normal réel','Suspect réel']).show()
px.histogram(scored, x='anomaly_score', color='is_anomaly', barmode='overlay').show()

## 13. Comparaison aux règles
Les règles utilisent au moins trois échecs, six tentatives ou une IP nouvelle avant 6 h. La sensibilité au seuil est descriptive ; le jeu de test ne doit pas servir à sélectionner un modèle final.

In [ ]:
comparison = pd.DataFrame(results).T[['precision','recall','f1']]
px.bar(comparison, barmode='group').show()
display(pd.DataFrame(sensitivity(train_logs, test_logs, bundle)).drop(columns='confusion_matrix'))

## 14. Conclusion
Les performances sont celles réellement affichées ci-dessus. Un résultat anormal appelle une
vérification, pas un blocage automatique. Les données simulées ne représentent pas tout le trafic réel.
La nouveauté au démarrage, la dérive temporelle, les faux positifs et les faux négatifs limitent ce prototype.
Une extension rigoureuse réserverait une validation temporelle distincte pour régler les paramètres.

Référence : [documentation officielle IsolationForest](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.IsolationForest.html).